# imports

In [21]:
%reload_ext autoreload
%autoreload 2

import torch
import sys
import os
# from torchmetrics.functional.pairwise import pairwise_cosine_similarity
import numpy as np
# from tqdm import tqdm
# from matplotlib import pyplot as plt
from datasets import load_dataset
import json
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification
)

ROOT_PATH = os.path.join(
    (os.path.dirname(os.path.abspath(''))),
)

# local imports
sys.path.insert(
    0,
    ROOT_PATH
)
# from evaluation.bright.retrievers import (
#     get_scores,
#     calculate_retrieval_metrics
# )
sys.path.pop(0)

DEBUG_NUMBER_EMBEDS = 100

# functions

# compute embeds for documents

In [3]:
long_context = False
dataset_source = 'xlangai/BRIGHT'
task = "theoremqa_theorems"
document_postfix = ''
cache_dir = os.path.join(ROOT_PATH, "evaluation", "bright", "cache")
if long_context:
    doc_pairs = load_dataset(dataset_source, 'long_documents'+document_postfix, cache_dir=cache_dir)[task]
else:
    doc_pairs = load_dataset(dataset_source, 'documents'+document_postfix, cache_dir=cache_dir)[task]

doc_ids = []
documents = []
for dp in doc_pairs:
    doc_ids.append(dp['id'])
    documents.append(dp['content'])

In [4]:
print(len(documents))
print(documents[0])

23839
\begin{definition}[Definition:Addition]
'''Addition''' is the basic operation $+$ everyone is familiar with.
For example:
:$2 + 3 = 5$
:$47 \cdotp 3 + 191\cdotp 4 = 238 \cdotp 7$
\end{definition}


In [5]:
config_dir = os.path.join(ROOT_PATH, "evaluation", "bright", "configs")
model_arg = "reasonir"
with open(os.path.join(config_dir,model_arg.split('_ckpt')[0].split('_bilevel')[0],f"{task}.json")) as f:
    config = json.load(f)
instructions = config['instructions']

In [6]:
customized_checkpoint = 'reasonir/ReasonIR-8B'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(customized_checkpoint, torch_dtype="auto", trust_remote_code=True)
model = AutoModel.from_pretrained(customized_checkpoint, torch_dtype="auto", trust_remote_code=True)
model.eval()
model.to(device)
query_instruction = instructions['query'].format(task=task)
doc_instruction = instructions['document']
# query_max_length = kwargs.get('query_max_length',32768)
# doc_max_length = kwargs.get('doc_max_length',32768)
query_max_length = 32768
doc_max_length = 32768
print("doc max length:",doc_max_length)
print("query max length:", query_max_length)
batch_size = 1

Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00,  9.76it/s]


doc max length: 32768
query max length: 32768


In [7]:
print(doc_instruction)
print(batch_size)
print(len(documents))
print(doc_max_length)

<|embed|>

1
23839
32768


In [22]:
# Override CUDA device count to 1 to control batch processing
torch.cuda.device_count = lambda: 1
doc_emb = model.encode(documents[:DEBUG_NUMBER_EMBEDS], instruction=doc_instruction, batch_size=batch_size, max_length=doc_max_length)

In [9]:
_, doc_emb_presaved = torch.load(os.path.join(ROOT_PATH, "evaluation", "bright", 'base_doc_emb.pkl'))

In [10]:
print(doc_emb_presaved.shape)

(23839, 4096)


In [23]:
print(doc_emb_presaved[0].mean())
print(doc_emb[0].mean())
print(doc_emb_presaved[42].mean())
print(doc_emb[42].mean())
print(doc_emb_presaved[99].mean())
print(doc_emb[99].mean())

0.00029349822
0.00029349822
5.2872783e-05
5.2872783e-05
5.7571804e-05
5.7571804e-05


In [24]:
print(doc_emb.shape)
assert np.isclose(doc_emb, doc_emb_presaved[:DEBUG_NUMBER_EMBEDS]).all()

# to pass this, make sure that batch_size is 1 (by default it is multiplied by the number of GPUs)
# Override CUDA device count to 1 to control batch processing
# torch.cuda.device_count = lambda: 1

(100, 4096)
